<a href="https://www.kaggle.com/code/sreyaroychowdhury/snntorch?scriptVersionId=335585952" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
!pip install snntorch -q

import os
import glob
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import snntorch as snn
import numpy as np

# Configurations matching 1st Order LIF paper
IMG_SIZE     = 64
NUM_STEPS    = 25
BATCH_SIZE   = 256  # For both T4 GPUs
EPOCHS       = 50
LR           = 5e-4
BETA         = 0.9
THRESHOLD    = 1.0
DROPOUT_P    = 0.5
HIDDEN_SIZE  = 512
NUM_CLASSES  = 2
SEED         = 42

torch.manual_seed(SEED)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Primary Master Device:', device)
print('Total Available GPUs:', torch.cuda.device_count())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.6/133.6 kB 7.3 MB/s eta 0:00:00
Primary Master Device: cuda:0
Total Available GPUs: 2


In [2]:
class DroNetCollisionDataset(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
        self.samples = []

        for seq_dir in sorted(glob.glob(os.path.join(root, '*'))):
            if not os.path.isdir(seq_dir):
                continue
            label_file = os.path.join(seq_dir, 'labels.txt')
            img_dir    = os.path.join(seq_dir, 'images')
            if not os.path.isfile(label_file) or not os.path.isdir(img_dir):
                continue

            with open(label_file, 'r') as f:
                labels = [int(l.strip()) for l in f if l.strip() != '']

            img_paths = sorted(
                glob.glob(os.path.join(img_dir, '*.jpg')) +
                glob.glob(os.path.join(img_dir, '*.png'))
            )

            for img_path, label in zip(img_paths, labels):
                self.samples.append((img_path, label))

        print(f'Loaded {len(self.samples)} samples from {root}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('L')
        if self.transform:
            img = self.transform(img)
        return img, label

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor() 
])

BASE = '/kaggle/input/datasets/sreyaroychowdhury/dronet-collision/collision_dataset'

train_dataset = DroNetCollisionDataset(os.path.join(BASE, 'training'), transform)
test_dataset  = DroNetCollisionDataset(os.path.join(BASE, 'testing'), transform)

num_neg, num_pos = 24968, 4894
total = num_neg + num_pos
class_weights = torch.FloatTensor([total / (2.0 * num_neg), total / (2.0 * num_pos)]).to(device)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

Loaded 29862 samples from /kaggle/input/datasets/sreyaroychowdhury/dronet-collision/collision_dataset/training
Loaded 1576 samples from /kaggle/input/datasets/sreyaroychowdhury/dronet-collision/collision_dataset/testing


In [3]:
class MultiGPULIFSNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Initialize linear projections with explicit bias support
        self.fc1 = nn.Linear(IMG_SIZE * IMG_SIZE, HIDDEN_SIZE, bias=True)
        self.fc2 = nn.Linear(HIDDEN_SIZE, NUM_CLASSES, bias=True)
        
        # SNN spiking parameters aligned with reset-to-zero paper requirements
        self.lif1 = snn.Leaky(beta=BETA, threshold=THRESHOLD, reset_mechanism="zero", learn_beta=True, learn_threshold=True)
        self.dropout = nn.Dropout(DROPOUT_P)
        self.lif2 = snn.Leaky(beta=BETA, threshold=THRESHOLD, reset_mechanism="zero", learn_beta=True, learn_threshold=True)
        
        # Configure secondary device targets
        self.gpu_ids = list(range(torch.cuda.device_count()))

    def forward(self, x):
        # Initialize membrane potentials
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        
        # Initialize refractory counters for batch size on the correct device
        batch_size = x.size(0)
        ref_count1 = torch.zeros(batch_size, HIDDEN_SIZE, device=x.device)
        ref_count2 = torch.zeros(batch_size, NUM_CLASSES, device=x.device)
        
        spk2_rec = []
        x_flattened = x.view(batch_size, -1)
        
        for step in range(NUM_STEPS):
            spike_in = torch.bernoulli(x_flattened)
            
            # --- LAYER 1 ---
            if len(self.gpu_ids) > 1 and self.training:
                inputs_scattered = nn.parallel.scatter(spike_in, self.gpu_ids)
                replicas = nn.parallel.replicate(self.fc1, self.gpu_ids)
                outputs_scattered = nn.parallel.parallel_apply(replicas, inputs_scattered)
                cur1 = nn.parallel.gather(outputs_scattered, target_device=self.gpu_ids[0])
            else:
                cur1 = self.fc1(spike_in)
            
            # Refractory Period: Mask input current to 0 for neurons in cool-down
            cur1 = torch.where(ref_count1 > 0, torch.zeros_like(cur1), cur1)
            
            spk1, mem1 = self.lif1(cur1, mem1)
            spk1 = self.dropout(spk1)
            
            # Update Refractory Counters for Layer 1
            ref_count1 = torch.where(spk1 > 0.5, torch.full_like(ref_count1, 5), ref_count1 - 1)
            ref_count1 = torch.clamp(ref_count1, min=0)
            
            # --- LAYER 2 ---
            if len(self.gpu_ids) > 1 and self.training:
                inputs_scattered2 = nn.parallel.scatter(spk1, self.gpu_ids)
                replicas2 = nn.parallel.replicate(self.fc2, self.gpu_ids)
                outputs_scattered2 = nn.parallel.parallel_apply(replicas2, inputs_scattered2)
                cur2 = nn.parallel.gather(outputs_scattered2, target_device=self.gpu_ids[0])
            else:
                cur2 = self.fc2(spk1)
                
            # Refractory Period: Mask input current for Layer 2
            cur2 = torch.where(ref_count2 > 0, torch.zeros_like(cur2), cur2)
            
            spk2, mem2 = self.lif2(cur2, mem2)
            spk2_rec.append(spk2)
            
            # Update Refractory Counters for Layer 2
            ref_count2 = torch.where(spk2 > 0.5, torch.full_like(ref_count2, 5), ref_count2 - 1)
            ref_count2 = torch.clamp(ref_count2, min=0)
            
        return torch.stack(spk2_rec, dim=0)

model = MultiGPULIFSNN().to(device)
print("Dual-GPU scattered module initialized safely.")

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

import snntorch.functional as sf
loss_fn = sf.ce_count_loss(weight=class_weights)

Dual-GPU scattered module initialized safely.


In [4]:
print("Beginning SNN dual-GPU split training loop with hardware constraints...")
for epoch in range(EPOCHS):
    model.train()
    loss_hist = []
    correct_train = 0
    total_train = 0
    
    for data, targets in train_loader:
        data, targets = data.to(device), targets.to(device)
        optimizer.zero_grad()
        spk_rec = model(data)
        
        loss_val = loss_fn(spk_rec, targets)
        loss_val.backward()
        optimizer.step()
        
        # --- HARDWARE CONSTRAINT CLAMPING ---
        with torch.no_grad():
            # Clamp Betas between 0.1 and 0.95 so they actually leak
            model.lif1.beta.clamp_(0.1, 0.95)
            model.lif2.beta.clamp_(0.1, 0.95)
            
            # Clamp Thresholds to be strictly positive (minimum 0.1)
            model.lif1.threshold.clamp_(min=0.1)
            model.lif2.threshold.clamp_(min=0.1)
        
        loss_hist.append(loss_val.item())
        _, idx = spk_rec.sum(dim=0).max(1)
        correct_train += (idx == targets).sum().item()
        total_train += targets.size(0)
        
    # Validation loop (rest of your eval code remains the same)
    model.eval()
    correct_test = 0
    total_test = 0
    with torch.no_grad():
        for data, targets in test_loader:
            data, targets = data.to(device), targets.to(device)
            spk_rec = model(data)
            _, idx = spk_rec.sum(dim=0).max(1)
            correct_test += (idx == targets).sum().item()
            total_test += targets.size(0)
            
    train_acc = (correct_train / total_train) * 100
    test_acc = (correct_test / total_test) * 100
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Loss: {np.mean(loss_hist):.4f} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")

Beginning SNN dual-GPU split training loop with hardware constraints...


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/linear.py:134: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:330.)
  return F.linear(input, self.weight, self.bias)


Epoch 01/50 | Loss: 0.6240 | Train Acc: 80.17% | Test Acc: 76.21%
Epoch 02/50 | Loss: 0.5199 | Train Acc: 80.51% | Test Acc: 80.14%
Epoch 03/50 | Loss: 0.4697 | Train Acc: 82.10% | Test Acc: 77.54%
Epoch 04/50 | Loss: 0.4221 | Train Acc: 84.64% | Test Acc: 74.05%
Epoch 05/50 | Loss: 0.3675 | Train Acc: 87.25% | Test Acc: 80.33%
Epoch 06/50 | Loss: 0.3411 | Train Acc: 87.94% | Test Acc: 84.26%
Epoch 07/50 | Loss: 0.3111 | Train Acc: 89.07% | Test Acc: 80.46%
Epoch 08/50 | Loss: 0.2862 | Train Acc: 90.24% | Test Acc: 80.71%
Epoch 09/50 | Loss: 0.2655 | Train Acc: 90.71% | Test Acc: 82.74%
Epoch 10/50 | Loss: 0.2583 | Train Acc: 91.21% | Test Acc: 80.84%
Epoch 11/50 | Loss: 0.2451 | Train Acc: 91.53% | Test Acc: 80.84%
Epoch 12/50 | Loss: 0.2325 | Train Acc: 91.93% | Test Acc: 81.98%
Epoch 13/50 | Loss: 0.2258 | Train Acc: 92.13% | Test Acc: 81.92%
Epoch 14/50 | Loss: 0.2239 | Train Acc: 92.09% | Test Acc: 79.82%
Epoch 15/50 | Loss: 0.2248 | Train Acc: 92.17% | Test Acc: 82.30%
Epoch 16/5

In [5]:
def export_param_to_q115_hex(tensor, filename):
    flat_data = tensor.detach().cpu().numpy().flatten()
    clipped_data = np.clip(flat_data, -1.0, 32767.0 / 32768.0)
    q115_int = np.round(clipped_data * 32768).astype(np.int32)
    
    with open(filename, 'w') as f:
        for val in q115_int:
            if val < 0:
                val = (1 << 16) + val
            f.write(f"{val:04X}\n")
    print(f"Successfully generated '{filename}' ({len(q115_int)} entries).")

export_param_to_q115_hex(model.fc1.weight, "layer1_weights_hex.txt")
export_param_to_q115_hex(model.fc2.weight, "layer2_weights_hex.txt")
export_param_to_q115_hex(model.fc1.bias,   "layer1_biases_hex.txt")
export_param_to_q115_hex(model.fc2.bias,   "layer2_biases_hex.txt")

export_param_to_q115_hex(model.lif1.threshold, "layer1_thresholds_hex.txt")
export_param_to_q115_hex(model.lif2.threshold, "layer2_thresholds_hex.txt")
export_param_to_q115_hex(model.lif1.beta,      "layer1_betas_hex.txt")
export_param_to_q115_hex(model.lif2.beta,      "layer2_betas_hex.txt")

Successfully generated 'layer1_weights_hex.txt' (2097152 entries).
Successfully generated 'layer2_weights_hex.txt' (1024 entries).
Successfully generated 'layer1_biases_hex.txt' (512 entries).
Successfully generated 'layer2_biases_hex.txt' (2 entries).
Successfully generated 'layer1_thresholds_hex.txt' (1 entries).
Successfully generated 'layer2_thresholds_hex.txt' (1 entries).
Successfully generated 'layer1_betas_hex.txt' (1 entries).
Successfully generated 'layer2_betas_hex.txt' (1 entries).


In [6]:
import os

def generate_true_split_file(root_dir, output_txt):
    entries = []
    collision_count = 0
    safe_count = 0
    
    # Get all sequence directories (e.g., GOPRO0387, DSCN2561)
    sequences = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
    
    for seq in sequences:
        seq_path = os.path.join(root_dir, seq)
        labels_file = os.path.join(seq_path, 'labels.txt')
        images_dir = os.path.join(seq_path, 'images')
        
        if not os.path.exists(labels_file) or not os.path.exists(images_dir):
            continue
            
        # Read the ground truth labels for this video sequence
        with open(labels_file, 'r') as lf:
            # Clean lines and split by whitespace or commas
            labels = [line.strip().split() for line in lf.readlines() if line.strip()]
            
        # Get all frames matching the labels sequentially
        frames = sorted([f for f in os.listdir(images_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        
        # Match each frame with its corresponding line in labels.txt
        for i, frame in enumerate(frames):
            if i < len(labels):
                # In DroNet, the collision label is typically the last column (0 for safe, 1 for collision)
                # Or a single value if it's a dedicated attribute file
                val = float(labels[i][-1])
                category = 'collision' if val >= 0.5 else 'safe'
                
                if category == 'collision':
                    collision_count += 1
                else:
                    safe_count += 1
                    
                relative_frame_path = os.path.join(seq, 'images', frame)
                entries.append(f"{relative_frame_path},{category}")
                
    with open(output_txt, 'w') as f:
        for entry in entries:
            f.write(f"{entry}\n")
            
    print(f"--- {os.path.basename(output_txt)} ---")
    print(f"Total frames: {len(entries)}")
    print(f"  Collision frames: {collision_count}")
    print(f"  Safe frames: {safe_count}\n")

train_path = '/kaggle/input/datasets/sreyaroychowdhury/dronet-collision/collision_dataset/training'
test_path = '/kaggle/input/datasets/sreyaroychowdhury/dronet-collision/collision_dataset/testing'

generate_true_split_file(train_path, '/kaggle/working/train_split.txt')
generate_true_split_file(test_path, '/kaggle/working/test_split.txt')

--- train_split.txt ---
Total frames: 29862
  Collision frames: 4894
  Safe frames: 24968

--- test_split.txt ---
Total frames: 1576
  Collision frames: 353
  Safe frames: 1223



In [7]:
# Run this after training to find the target parameters for the HW approximation
print("--- Learned Parameters (Post-Phase 1) ---")
print(f"Layer 1 Beta Mean: {model.lif1.beta.data.mean().item():.4f}")
print(f"Layer 2 Beta Mean: {model.lif2.beta.data.mean().item():.4f}")
print(f"Layer 1 Threshold Mean: {model.lif1.threshold.data.mean().item():.4f}")
print(f"Layer 2 Threshold Mean: {model.lif2.threshold.data.mean().item():.4f}")

--- Learned Parameters (Post-Phase 1) ---
Layer 1 Beta Mean: 0.9500
Layer 2 Beta Mean: 0.9500
Layer 1 Threshold Mean: 0.1000
Layer 2 Threshold Mean: 0.1000
